In [4]:
from google.colab import drive
drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
os.chdir("/content")
print(os.getcwd())


/content


In [5]:
from pathlib import Path
import re
import json
from collections import Counter

import pandas as pd
from PIL import Image, UnidentifiedImageError
from IPython.display import display

DRIVE_ROOT = Path("/content/drive/MyDrive")
PIPELINE_ROOT = DRIVE_ROOT / "task_driven_video_pipeline"
KAGGLE_V1_ROOT = PIPELINE_ROOT / "kaggle_v1"

RAW_CLEAN_ROOT = KAGGLE_V1_ROOT / "raw_clean"
AUDIT_ROOT = KAGGLE_V1_ROOT / "audit"

AUDIT_ROOT.mkdir(parents=True, exist_ok=True)

MANIFEST_CSV = AUDIT_ROOT / "manifest.csv"
BROKEN_CSV = AUDIT_ROOT / "broken_files.csv"
CLASS_COUNTS_CSV = AUDIT_ROOT / "class_counts.csv"
SUBJECT_SUMMARY_CSV = AUDIT_ROOT / "subject_summary.csv"
SIZE_DISTRIBUTION_CSV = AUDIT_ROOT / "image_size_distribution.csv"
SPLIT_CONFIG_JSON = AUDIT_ROOT / "split_config.json"
MANIFEST_WITH_SPLIT_CSV = AUDIT_ROOT / "manifest_with_split.csv"
SPLIT_COUNTS_CSV = AUDIT_ROOT / "split_class_counts.csv"

assert RAW_CLEAN_ROOT.exists(), f"Missing RAW_CLEAN_ROOT: {RAW_CLEAN_ROOT}"
assert (RAW_CLEAN_ROOT / "open").exists(), f"Missing open folder: {RAW_CLEAN_ROOT / 'open'}"
assert (RAW_CLEAN_ROOT / "closed").exists(), f"Missing closed folder: {RAW_CLEAN_ROOT / 'closed'}"

print("RAW_CLEAN_ROOT:", RAW_CLEAN_ROOT)
print("AUDIT_ROOT:", AUDIT_ROOT)


RAW_CLEAN_ROOT: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/raw_clean
AUDIT_ROOT: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/audit


In [6]:
OPEN_DIR = RAW_CLEAN_ROOT / "open"
CLOSED_DIR = RAW_CLEAN_ROOT / "closed"

VALID_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}
SUBJECT_RE = re.compile(r"^(s\d+)", re.IGNORECASE)

def parse_subject_id(filename: str) -> str:
    match = SUBJECT_RE.match(filename)
    if match:
        return match.group(1).lower()
    return "unknown"

def iter_image_files(class_dir: Path):
    for p in class_dir.rglob("*"):
        if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS and not p.name.startswith("."):
            yield p

open_files = list(iter_image_files(OPEN_DIR))
closed_files = list(iter_image_files(CLOSED_DIR))

print("Open files found:", len(open_files))
print("Closed files found:", len(closed_files))
print("Total files found:", len(open_files) + len(closed_files))


Open files found: 16963
Closed files found: 24000
Total files found: 40963


In [7]:
from collections import Counter

def inspect_class_dir(class_dir: Path, label: str):
    all_files = [p for p in class_dir.rglob("*") if p.is_file()]
    valid_files = [
        p for p in all_files
        if p.suffix.lower() in VALID_EXTENSIONS and not p.name.startswith(".")
    ]
    hidden_files = [p for p in all_files if p.name.startswith(".")]
    invalid_ext = [p for p in all_files if p.suffix.lower() not in VALID_EXTENSIONS]

    print(f"\n=== {label.upper()} ===")
    print("all files:", len(all_files))
    print("valid files:", len(valid_files))
    print("hidden files:", len(hidden_files))
    print("unsupported extension files:", len(invalid_ext))
    print("extension breakdown:", Counter((p.suffix.lower() or "<no_ext>") for p in all_files).most_common(10))
    print("subject sample:", Counter(parse_subject_id(p.name) for p in valid_files).most_common(15))

inspect_class_dir(OPEN_DIR, "open")
inspect_class_dir(CLOSED_DIR, "closed")



=== OPEN ===
all files: 16963
valid files: 16963
hidden files: 0
unsupported extension files: 0
extension breakdown: [('.png', 16963)]
subject sample: [('s0014', 5116), ('s0012', 4230), ('s0019', 3266), ('s0001', 1185), ('s0018', 721), ('s0013', 630), ('s0020', 467), ('s0015', 348), ('s0017', 329), ('s0016', 173), ('s0011', 140), ('s0003', 120), ('s0002', 107), ('s0010', 68), ('s0005', 28)]

=== CLOSED ===
all files: 24000
valid files: 24000
hidden files: 0
unsupported extension files: 0
extension breakdown: [('.png', 24000)]
subject sample: [('s0012', 4498), ('s0014', 3768), ('s0019', 2905), ('s0001', 1742), ('s0016', 1704), ('s0011', 1110), ('s0004', 1050), ('s0002', 1006), ('s0006', 955), ('s0008', 832), ('s0015', 776), ('s0005', 708), ('s0007', 613), ('s0003', 554), ('s0017', 520)]


In [9]:
records = []
broken_records = []

all_sources = [
    ("open", open_files),
    ("closed", closed_files),
]

total_files = sum(len(files) for _, files in all_sources)
seen = 0

for class_name, files in all_sources:
    for p in files:
        seen += 1
        if seen == 1 or seen % 2000 == 0:
            print(f"Scanning {seen}/{total_files}")

        subject_id = parse_subject_id(p.name)

        try:
            with Image.open(p) as img:
                width, height = img.size
                mode = img.mode
        except (UnidentifiedImageError, OSError, ValueError) as e:
            broken_records.append({
                "path": str(p),
                "class_name": class_name,
                "file_name": p.name,
                "subject_id": subject_id,
                "error": str(e),
            })
            continue

        records.append({
            "path": str(p),
            "class_name": class_name,
            "file_name": p.name,
            "subject_id": subject_id,
            "width": int(width),
            "height": int(height),
            "mode": mode,
        })

manifest_df = pd.DataFrame(records)
broken_df = pd.DataFrame(broken_records)

manifest_df.to_csv(MANIFEST_CSV, index=False)
broken_df.to_csv(BROKEN_CSV, index=False)

print("Usable images:", len(manifest_df))
print("Broken images:", len(broken_df))
print("Saved manifest to:", MANIFEST_CSV)
print("Saved broken file report to:", BROKEN_CSV)


Scanning 1/40963
Scanning 2000/40963
Scanning 4000/40963
Scanning 6000/40963
Scanning 8000/40963
Scanning 10000/40963
Scanning 12000/40963
Scanning 14000/40963
Scanning 16000/40963
Scanning 18000/40963
Scanning 20000/40963
Scanning 22000/40963
Scanning 24000/40963


KeyboardInterrupt: 

In [11]:
import random
from collections import defaultdict
from pathlib import Path
from PIL import Image, UnidentifiedImageError
import pandas as pd

TARGET_TOTAL = 20000
TARGET_PER_CLASS = TARGET_TOTAL // 2
SEED = 42
PROGRESS_EVERY = 250

if "records" not in globals():
    raise RuntimeError("Interrupt the old cell without restarting the runtime, then run this.")

if "broken_records" not in globals():
    broken_records = []

OUTPUT_DIR = Path(MANIFEST_CSV).parent
SUBSET_MANIFEST_CSV = OUTPUT_DIR / "manifest_subject_class_balanced_20k.csv"
SUBSET_SUMMARY_CSV = OUTPUT_DIR / "manifest_subject_class_balanced_20k_subject_summary.csv"
SCANNED_SNAPSHOT_CSV = OUTPUT_DIR / "manifest_scanned_snapshot.csv"
BROKEN_SNAPSHOT_CSV = OUTPUT_DIR / "broken_scanned_snapshot.csv"

def group_by_subject(paths):
    grouped = defaultdict(list)
    for p in paths:
        grouped[parse_subject_id(p.name)].append(p)
    return {sid: sorted(items) for sid, items in grouped.items()}

def allocate_even_pairs(subject_capacities, target_pairs):
    allocation = {sid: 0 for sid in subject_capacities}
    active = [sid for sid, cap in sorted(subject_capacities.items()) if cap > 0]
    allocated = 0
    while active and allocated < target_pairs:
        next_active = []
        for sid in active:
            if allocated >= target_pairs:
                break
            if allocation[sid] < subject_capacities[sid]:
                allocation[sid] += 1
                allocated += 1
            if allocation[sid] < subject_capacities[sid]:
                next_active.append(sid)
        active = next_active
    return allocation

def scan_one(path, class_name, valid_lookup, broken_lookup):
    subject_id = parse_subject_id(path.name)
    try:
        with Image.open(path) as img:
            width, height = img.size
            mode = img.mode
        row = {
            "path": str(path),
            "class_name": class_name,
            "file_name": path.name,
            "subject_id": subject_id,
            "width": int(width),
            "height": int(height),
            "mode": mode,
        }
        records.append(row)
        valid_lookup[str(path)] = row
        return row
    except (UnidentifiedImageError, OSError, ValueError) as e:
        broken_records.append({
            "path": str(path),
            "class_name": class_name,
            "file_name": path.name,
            "subject_id": subject_id,
            "error": str(e),
        })
        broken_lookup.add(str(path))
        return None

def take_k_valid_rows(pool, class_name, k, rng, valid_lookup, broken_lookup, stats):
    shuffled = list(pool)
    rng.shuffle(shuffled)
    chosen = []

    for p in shuffled:
        if len(chosen) >= k:
            break

        key = str(p)
        cached = valid_lookup.get(key)
        if cached is not None:
            chosen.append(cached)
            stats["reused"] += 1
            continue

        if key in broken_lookup:
            continue

        row = scan_one(p, class_name, valid_lookup, broken_lookup)
        stats["new_scans"] += 1

        if row is not None:
            chosen.append(row)
            stats["new_valid"] += 1
        else:
            stats["new_broken"] += 1

        if stats["new_scans"] == 1 or stats["new_scans"] % PROGRESS_EVERY == 0:
            print(
                f"New scans: {stats['new_scans']} | reused: {stats['reused']} | "
                f"new valid: {stats['new_valid']} | new broken: {stats['new_broken']}"
            )

    if len(chosen) < k:
        raise RuntimeError(
            f"Subject/class pool exhausted for {class_name}. "
            f"Needed {k}, got {len(chosen)} valid files."
        )

    return chosen

valid_lookup = {row["path"]: row for row in records if "path" in row}
broken_lookup = {row["path"] for row in broken_records if "path" in row}

open_by_subject = group_by_subject(open_files)
closed_by_subject = group_by_subject(closed_files)

subject_capacities = {}
for sid in sorted(set(open_by_subject) & set(closed_by_subject)):
    cap = min(len(open_by_subject[sid]), len(closed_by_subject[sid]))
    if cap > 0:
        subject_capacities[sid] = cap

max_pairs_from_names = sum(subject_capacities.values())
target_pairs = min(TARGET_PER_CLASS, max_pairs_from_names)

print("Cached usable records:", len(valid_lookup))
print("Cached broken records:", len(broken_lookup))
print("Max balanced subject-pairs from filenames:", max_pairs_from_names)
print("Requested pairs:", TARGET_PER_CLASS)
print("Target pairs to build:", target_pairs)

if target_pairs == 0:
    raise RuntimeError("No shared subjects with both open and closed images were found.")

allocation = allocate_even_pairs(subject_capacities, target_pairs)
selected_subjects = [sid for sid, k in allocation.items() if k > 0]

rng = random.Random(SEED)
stats = {"reused": 0, "new_scans": 0, "new_valid": 0, "new_broken": 0}
selected_rows = []
summary_rows = []

for idx, sid in enumerate(selected_subjects, start=1):
    k = allocation[sid]
    selected_rows.extend(
        take_k_valid_rows(open_by_subject[sid], "open", k, rng, valid_lookup, broken_lookup, stats)
    )
    selected_rows.extend(
        take_k_valid_rows(closed_by_subject[sid], "closed", k, rng, valid_lookup, broken_lookup, stats)
    )

    summary_rows.append({
        "subject_id": sid,
        "selected_per_class": k,
        "selected_open": k,
        "selected_closed": k,
        "available_open_files": len(open_by_subject[sid]),
        "available_closed_files": len(closed_by_subject[sid]),
        "max_pairs_from_filenames": subject_capacities[sid],
    })

    if idx == 1 or idx % 25 == 0 or idx == len(selected_subjects):
        print(f"Subjects processed: {idx}/{len(selected_subjects)} | selected rows: {len(selected_rows)}")

subset_df = pd.DataFrame(selected_rows).sort_values(
    ["subject_id", "class_name", "file_name", "path"]
).reset_index(drop=True)

summary_df = pd.DataFrame(summary_rows).sort_values(
    ["selected_per_class", "subject_id"],
    ascending=[False, True],
).reset_index(drop=True)

pd.DataFrame(records).to_csv(SCANNED_SNAPSHOT_CSV, index=False)
pd.DataFrame(broken_records).to_csv(BROKEN_SNAPSHOT_CSV, index=False)
subset_df.to_csv(SUBSET_MANIFEST_CSV, index=False)
summary_df.to_csv(SUBSET_SUMMARY_CSV, index=False)

class_counts = subset_df["class_name"].value_counts().to_dict()

print()
print("Final subset counts")
print("open:", class_counts.get("open", 0))
print("closed:", class_counts.get("closed", 0))
print("total:", len(subset_df))
print("subjects used:", summary_df["subject_id"].nunique())
print("reused cached records:", stats["reused"])
print("newly scanned files:", stats["new_scans"])
print("saved subset manifest to:", SUBSET_MANIFEST_CSV)
print("saved subject summary to:", SUBSET_SUMMARY_CSV)


Cached usable records: 24554
Cached broken records: 0
Max balanced subject-pairs from filenames: 13792
Requested pairs: 10000
Target pairs to build: 10000
New scans: 1 | reused: 1185 | new valid: 1 | new broken: 0
New scans: 250 | reused: 1185 | new valid: 250 | new broken: 0
New scans: 500 | reused: 1185 | new valid: 500 | new broken: 0
New scans: 750 | reused: 1185 | new valid: 750 | new broken: 0
New scans: 1000 | reused: 1185 | new valid: 1000 | new broken: 0
Subjects processed: 1/16 | selected rows: 2370
New scans: 1250 | reused: 1292 | new valid: 1250 | new broken: 0
New scans: 1500 | reused: 1543 | new valid: 1500 | new broken: 0
New scans: 1750 | reused: 4054 | new valid: 1750 | new broken: 0
New scans: 2000 | reused: 4054 | new valid: 2000 | new broken: 0
New scans: 2250 | reused: 4054 | new valid: 2250 | new broken: 0
New scans: 2500 | reused: 4054 | new valid: 2500 | new broken: 0
New scans: 2750 | reused: 4054 | new valid: 2750 | new broken: 0
New scans: 3000 | reused: 4054

In [12]:
# Use the selected 20k subset as the manifest for all later analysis cells
manifest_df = subset_df.copy()

MANIFEST_CSV = SUBSET_MANIFEST_CSV
CLASS_COUNTS_CSV = AUDIT_ROOT / "class_counts_subject_class_balanced_20k.csv"
SUBJECT_SUMMARY_CSV = AUDIT_ROOT / "subject_summary_subject_class_balanced_20k.csv"
SIZE_DISTRIBUTION_CSV = AUDIT_ROOT / "image_size_distribution_subject_class_balanced_20k.csv"
MANIFEST_WITH_SPLIT_CSV = AUDIT_ROOT / "manifest_with_split_subject_class_balanced_20k.csv"
SPLIT_COUNTS_CSV = AUDIT_ROOT / "split_class_counts_subject_class_balanced_20k.csv"
SPLIT_CONFIG_JSON = AUDIT_ROOT / "split_config_subject_class_balanced_20k.json"

print("Now using subset manifest for downstream analysis:")
print(MANIFEST_CSV)
print("Rows in manifest_df:", len(manifest_df))
print(manifest_df["class_name"].value_counts())


Now using subset manifest for downstream analysis:
/content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/audit/manifest_subject_class_balanced_20k.csv
Rows in manifest_df: 20000
class_name
closed    10000
open      10000
Name: count, dtype: int64


In [13]:
class_counts = (
    manifest_df["class_name"]
    .value_counts()
    .rename_axis("class_name")
    .reset_index(name="count")
    .sort_values("class_name")
)

class_counts.to_csv(CLASS_COUNTS_CSV, index=False)

display(class_counts)
print("Saved class counts to:", CLASS_COUNTS_CSV)


,class_name,count
0,closed,10000
1,open,10000


Saved class counts to: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/audit/class_counts_subject_class_balanced_20k.csv


In [14]:
subject_summary = (
    manifest_df
    .pivot_table(index="subject_id", columns="class_name", values="file_name", aggfunc="count", fill_value=0)
    .reset_index()
)

if "open" not in subject_summary.columns:
    subject_summary["open"] = 0
if "closed" not in subject_summary.columns:
    subject_summary["closed"] = 0

subject_summary["total"] = subject_summary["open"] + subject_summary["closed"]
subject_summary["in_both_classes"] = (subject_summary["open"] > 0) & (subject_summary["closed"] > 0)
subject_summary["class_presence"] = subject_summary.apply(
    lambda row: (
        "both" if row["open"] > 0 and row["closed"] > 0
        else "open_only" if row["open"] > 0
        else "closed_only" if row["closed"] > 0
        else "none"
    ),
    axis=1,
)

subject_summary = subject_summary.sort_values(["total", "subject_id"], ascending=[False, True]).reset_index(drop=True)
subject_summary.to_csv(SUBJECT_SUMMARY_CSV, index=False)

display(subject_summary.head(30))
print("Saved subject summary to:", SUBJECT_SUMMARY_CSV)


class_name,subject_id,closed,open,total,in_both_classes,class_presence
0,s0012,2371,2371,4742,True,both
1,s0014,2370,2370,4740,True,both
2,s0019,2370,2370,4740,True,both
3,s0001,1185,1185,2370,True,both
4,s0015,348,348,696,True,both
5,s0013,331,331,662,True,both
6,s0017,329,329,658,True,both
7,s0016,173,173,346,True,both
8,s0011,140,140,280,True,both
9,s0003,120,120,240,True,both


Saved subject summary to: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/audit/subject_summary_subject_class_balanced_20k.csv


In [15]:
both_subjects = subject_summary.loc[subject_summary["class_presence"] == "both", "subject_id"].tolist()
open_only_subjects = subject_summary.loc[subject_summary["class_presence"] == "open_only", "subject_id"].tolist()
closed_only_subjects = subject_summary.loc[subject_summary["class_presence"] == "closed_only", "subject_id"].tolist()

print("Subjects present in both classes:", len(both_subjects))
print(both_subjects)
print()

print("Open-only subjects:", len(open_only_subjects))
print(open_only_subjects)
print()

print("Closed-only subjects:", len(closed_only_subjects))
print(closed_only_subjects)


Subjects present in both classes: 16
['s0012', 's0014', 's0019', 's0001', 's0015', 's0013', 's0017', 's0016', 's0011', 's0003', 's0002', 's0010', 's0005', 's0020', 's0009', 's0007']

Open-only subjects: 0
[]

Closed-only subjects: 0
[]


In [16]:
size_distribution = (
    manifest_df
    .groupby(["width", "height"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

size_distribution.to_csv(SIZE_DISTRIBUTION_CSV, index=False)

display(size_distribution.head(20))
print("Saved image size distribution to:", SIZE_DISTRIBUTION_CSV)


,width,height,count
0,84,84,876
1,83,83,807
2,86,86,802
3,85,85,753
4,82,82,728
5,88,88,720
6,87,87,705
7,89,89,619
8,81,81,605
9,80,80,567


Saved image size distribution to: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/audit/image_size_distribution_subject_class_balanced_20k.csv


In [17]:
print("Files with unknown subject IDs:", (manifest_df["subject_id"] == "unknown").sum())
print("Unique subjects total:", manifest_df["subject_id"].nunique())


Files with unknown subject IDs: 0
Unique subjects total: 16


In [18]:
VAL_SUBJECT_COUNT = 4
TEST_SUBJECT_COUNT = 5
MIN_PER_CLASS_FOR_EVAL = 50
INCLUDE_ONE_CLASS_ONLY_IN_TRAIN = True
EXCLUDE_UNKNOWN_SUBJECTS = True

both_df = subject_summary[subject_summary["class_presence"] == "both"].copy()

eligible_eval_df = both_df[
    (both_df["open"] >= MIN_PER_CLASS_FOR_EVAL) &
    (both_df["closed"] >= MIN_PER_CLASS_FOR_EVAL)
].copy()

if len(eligible_eval_df) < (VAL_SUBJECT_COUNT + TEST_SUBJECT_COUNT):
    print("Not enough subjects passed the MIN_PER_CLASS_FOR_EVAL filter. Falling back to all both-class subjects.")
    eligible_eval_df = both_df.copy()

eligible_eval_df = eligible_eval_df.sort_values(["total", "subject_id"], ascending=[False, True]).reset_index(drop=True)

display(eligible_eval_df[["subject_id", "open", "closed", "total"]])


class_name,subject_id,open,closed,total
0,s0012,2371,2371,4742
1,s0014,2370,2370,4740
2,s0019,2370,2370,4740
3,s0001,1185,1185,2370
4,s0015,348,348,696
5,s0013,331,331,662
6,s0017,329,329,658
7,s0016,173,173,346
8,s0011,140,140,280
9,s0003,120,120,240


In [19]:
def greedy_pick_subjects(subject_df: pd.DataFrame, n_pick: int):
    subject_df = subject_df.copy().reset_index(drop=True)
    chosen = []

    current_open = 0
    current_closed = 0

    total_open = int(subject_df["open"].sum())
    total_closed = int(subject_df["closed"].sum())

    for step in range(n_pick):
        target_open = total_open * (step + 1) / n_pick
        target_closed = total_closed * (step + 1) / n_pick

        best_row_idx = None
        best_score = None

        for idx, row in subject_df.iterrows():
            new_open = current_open + int(row["open"])
            new_closed = current_closed + int(row["closed"])

            score = (
                abs(new_open - target_open)
                + abs(new_closed - target_closed)
                + 0.25 * abs(new_open - new_closed)
            )

            score -= 0.0001 * int(row["total"])

            if best_score is None or score < best_score:
                best_score = score
                best_row_idx = idx

        picked = subject_df.loc[best_row_idx].to_dict()
        chosen.append(picked)

        current_open += int(picked["open"])
        current_closed += int(picked["closed"])
        subject_df = subject_df.drop(best_row_idx).reset_index(drop=True)

    return chosen, subject_df

recommended_test_rows, remaining_after_test = greedy_pick_subjects(eligible_eval_df, TEST_SUBJECT_COUNT)
recommended_val_rows, remaining_after_val = greedy_pick_subjects(remaining_after_test, VAL_SUBJECT_COUNT)

RECOMMENDED_TEST_SUBJECTS = sorted([row["subject_id"] for row in recommended_test_rows])
RECOMMENDED_VAL_SUBJECTS = sorted([row["subject_id"] for row in recommended_val_rows])

print("Recommended test subjects:", RECOMMENDED_TEST_SUBJECTS)
print("Recommended val subjects:", RECOMMENDED_VAL_SUBJECTS)


Recommended test subjects: ['s0001', 's0012', 's0014', 's0015', 's0019']
Recommended val subjects: ['s0011', 's0013', 's0016', 's0017']


In [20]:
# If you want, you can manually edit these two lists after seeing the recommendation.
VAL_SUBJECTS = RECOMMENDED_VAL_SUBJECTS.copy()
TEST_SUBJECTS = RECOMMENDED_TEST_SUBJECTS.copy()

print("VAL_SUBJECTS:", VAL_SUBJECTS)
print("TEST_SUBJECTS:", TEST_SUBJECTS)


VAL_SUBJECTS: ['s0011', 's0013', 's0016', 's0017']
TEST_SUBJECTS: ['s0001', 's0012', 's0014', 's0015', 's0019']


In [21]:
all_both_subjects = set(both_subjects)
all_one_class_only_subjects = set(open_only_subjects + closed_only_subjects)

train_subjects = sorted(all_both_subjects - set(VAL_SUBJECTS) - set(TEST_SUBJECTS))

if INCLUDE_ONE_CLASS_ONLY_IN_TRAIN:
    train_subjects = sorted(set(train_subjects) | all_one_class_only_subjects)

def assign_split(subject_id: str) -> str:
    if EXCLUDE_UNKNOWN_SUBJECTS and subject_id == "unknown":
        return "excluded_unknown"
    if subject_id in TEST_SUBJECTS:
        return "test"
    if subject_id in VAL_SUBJECTS:
        return "val"
    if subject_id in train_subjects:
        return "train"
    return "excluded"

manifest_with_split = manifest_df.copy()
manifest_with_split["split"] = manifest_with_split["subject_id"].apply(assign_split)

manifest_with_split.to_csv(MANIFEST_WITH_SPLIT_CSV, index=False)

split_class_counts = (
    manifest_with_split
    .groupby(["split", "class_name"])
    .size()
    .reset_index(name="count")
    .sort_values(["split", "class_name"])
)

split_class_counts.to_csv(SPLIT_COUNTS_CSV, index=False)

display(split_class_counts)
print("Saved split manifest to:", MANIFEST_WITH_SPLIT_CSV)
print("Saved split counts to:", SPLIT_COUNTS_CSV)


,split,class_name,count
0,test,closed,8644
1,test,open,8644
2,train,closed,383
3,train,open,383
4,val,closed,973
5,val,open,973


Saved split manifest to: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/audit/manifest_with_split_subject_class_balanced_20k.csv
Saved split counts to: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/audit/split_class_counts_subject_class_balanced_20k.csv


In [22]:
split_config = {
    "raw_clean_root": str(RAW_CLEAN_ROOT),
    "val_subject_count": VAL_SUBJECT_COUNT,
    "test_subject_count": TEST_SUBJECT_COUNT,
    "min_per_class_for_eval": MIN_PER_CLASS_FOR_EVAL,
    "include_one_class_only_in_train": INCLUDE_ONE_CLASS_ONLY_IN_TRAIN,
    "exclude_unknown_subjects": EXCLUDE_UNKNOWN_SUBJECTS,
    "recommended_val_subjects": RECOMMENDED_VAL_SUBJECTS,
    "recommended_test_subjects": RECOMMENDED_TEST_SUBJECTS,
    "final_val_subjects": VAL_SUBJECTS,
    "final_test_subjects": TEST_SUBJECTS,
    "final_train_subjects": train_subjects,
    "both_class_subjects": both_subjects,
    "open_only_subjects": open_only_subjects,
    "closed_only_subjects": closed_only_subjects,
}

with open(SPLIT_CONFIG_JSON, "w", encoding="utf-8") as f:
    json.dump(split_config, f, indent=2)

print("Saved split config to:", SPLIT_CONFIG_JSON)
print()
print("Train subjects:", len(train_subjects))
print(train_subjects)
print()
print("Val subjects:", len(VAL_SUBJECTS))
print(VAL_SUBJECTS)
print()
print("Test subjects:", len(TEST_SUBJECTS))
print(TEST_SUBJECTS)


Saved split config to: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/audit/split_config_subject_class_balanced_20k.json

Train subjects: 7
['s0002', 's0003', 's0005', 's0007', 's0009', 's0010', 's0020']

Val subjects: 4
['s0011', 's0013', 's0016', 's0017']

Test subjects: 5
['s0001', 's0012', 's0014', 's0015', 's0019']


In [23]:
print("Phase 1 complete.")
print("Manifest:", MANIFEST_CSV)
print("Broken file report:", BROKEN_CSV)
print("Class counts:", CLASS_COUNTS_CSV)
print("Subject summary:", SUBJECT_SUMMARY_CSV)
print("Image size distribution:", SIZE_DISTRIBUTION_CSV)
print("Manifest with split:", MANIFEST_WITH_SPLIT_CSV)
print("Split config:", SPLIT_CONFIG_JSON)


Phase 1 complete.
Manifest: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/audit/manifest_subject_class_balanced_20k.csv
Broken file report: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/audit/broken_files.csv
Class counts: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/audit/class_counts_subject_class_balanced_20k.csv
Subject summary: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/audit/subject_summary_subject_class_balanced_20k.csv
Image size distribution: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/audit/image_size_distribution_subject_class_balanced_20k.csv
Manifest with split: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/audit/manifest_with_split_subject_class_balanced_20k.csv
Split config: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/audit/split_config_subject_class_balanced_20k.json


In [ ]:
import shutil
from pathlib import Path
import pandas as pd

SUBSET_OUTPUT_ROOT = PIPELINE_ROOT / "kaggle_v1" / "raw_clean_subject_class_balanced_20k"
COPIED_REPORT_CSV = AUDIT_ROOT / "copied_files_subject_class_balanced_20k.csv"
PROGRESS_EVERY = 500

subset_manifest_path = Path(SUBSET_MANIFEST_CSV)
if not subset_manifest_path.exists():
    raise FileNotFoundError(f"Subset manifest not found: {subset_manifest_path}")

subset_df = pd.read_csv(subset_manifest_path)

required_cols = {"path", "class_name", "file_name", "subject_id"}
missing_cols = required_cols - set(subset_df.columns)
if missing_cols:
    raise ValueError(f"Subset manifest is missing columns: {sorted(missing_cols)}")

if SUBSET_OUTPUT_ROOT.exists() and any(SUBSET_OUTPUT_ROOT.iterdir()):
    raise FileExistsError(
        f"Output folder already exists and is not empty: {SUBSET_OUTPUT_ROOT}\n"
        "Choose a new folder name or delete the old one first."
    )

open_root = OPEN_DIR.resolve()
closed_root = CLOSED_DIR.resolve()

(SUBSET_OUTPUT_ROOT / "open").mkdir(parents=True, exist_ok=True)
(SUBSET_OUTPUT_ROOT / "closed").mkdir(parents=True, exist_ok=True)

copied_rows = []
seen_destinations = set()

for idx, row in enumerate(subset_df.to_dict("records"), start=1):
    src = Path(row["path"]).resolve()
    class_name = str(row["class_name"]).strip().lower()

    if class_name == "open":
        relative_path = src.relative_to(open_root)
    elif class_name == "closed":
        relative_path = src.relative_to(closed_root)
    else:
        raise ValueError(f"Unexpected class_name: {class_name}")

    dest = SUBSET_OUTPUT_ROOT / class_name / relative_path

    if str(dest) in seen_destinations:
        raise RuntimeError(f"Duplicate destination detected: {dest}")
    seen_destinations.add(str(dest))

    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dest)

    copied_rows.append({
        "source_path": str(src),
        "destination_path": str(dest),
        "class_name": class_name,
        "subject_id": row["subject_id"],
        "file_name": row["file_name"],
    })

    if idx == 1 or idx % PROGRESS_EVERY == 0 or idx == len(subset_df):
        print(f"Copied {idx}/{len(subset_df)}")

copied_df = pd.DataFrame(copied_rows)
copied_df.to_csv(COPIED_REPORT_CSV, index=False)

print()
print("Done")
print("Output root:", SUBSET_OUTPUT_ROOT)
print("Open count:", (copied_df["class_name"] == "open").sum())
print("Closed count:", (copied_df["class_name"] == "closed").sum())
print("Total count:", len(copied_df))
print("Saved copy report to:", COPIED_REPORT_CSV)


Copied 1/20000
Copied 500/20000
Copied 1000/20000
Copied 1500/20000
Copied 2000/20000
Copied 2500/20000
Copied 3000/20000
Copied 3500/20000
Copied 4000/20000
Copied 4500/20000
Copied 5000/20000
Copied 5500/20000
Copied 6000/20000
Copied 6500/20000
